In [4]:
from google.colab import userdata, drive
import os

# Repo already cloned, just navigate into it
%cd /content/musicandmemory

# Pull latest changes in case teammates pushed updates
token = userdata.get("GITHUB_TOKEN")
!git pull https://{token}@github.com/anikanb-32/musicandmemory.git

/content/musicandmemory
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.10 KiB | 1.10 MiB/s, done.
From https://github.com/anikanb-32/musicandmemory
 * branch            HEAD       -> FETCH_HEAD
Updating 7cc1a78..242b339
Fast-forward
 notebooks/02_data_pipeline.ipynb | 110 ++++++++++++++++++++++++++++++++-------
 1 file changed, 90 insertions(+), 20 deletions(-)


In [5]:
drive.mount('/content/drive')
!pip install -r requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.4/404.4 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langcha

In [7]:
# load APIs

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["SPOTIFY_CLIENT_ID"] = userdata.get("SPOTIFY_CLIENT_ID")
os.environ["SPOTIFY_CLIENT_SECRET"] = userdata.get("SPOTIFY_CLIENT_SECRET")

DATA_PATH = '/content/drive/MyDrive/'

In [8]:
# load billboard data

import pandas as pd

df_billboard = pd.read_csv(DATA_PATH + "charts.csv")
print(f"Total rows: {len(df_billboard)}")
print(f"Columns: {list(df_billboard.columns)}")
df_billboard.head()

Total rows: 330087
Columns: ['date', 'rank', 'song', 'artist', 'last-week', 'peak-rank', 'weeks-on-board']


,date,rank,song,artist,last-week,peak-rank,weeks-on-board
0,2021-11-06,1,Easy On Me,Adele,1.0,1,3
1,2021-11-06,2,Stay,The Kid LAROI & Justin Bieber,2.0,1,16
2,2021-11-06,3,Industry Baby,Lil Nas X & Jack Harlow,3.0,1,14
3,2021-11-06,4,Fancy Like,Walker Hayes,4.0,3,19
4,2021-11-06,5,Bad Habits,Ed Sheeran,5.0,2,18


In [9]:
# cleaning data

# Standardize column names - fix spaces AND hyphens
df_billboard.columns = [col.strip().lower().replace(" ", "_").replace("-", "_") for col in df_billboard.columns]

# Convert date and extract year
df_billboard['date'] = pd.to_datetime(df_billboard['date'])
df_billboard['year'] = df_billboard['date'].dt.year

# Deduplicate keeping best chart position per song
df_deduped = (
    df_billboard
    .sort_values('peak_rank')
    .drop_duplicates(subset=['song', 'artist'], keep='first')
    .reset_index(drop=True)
)

print(f"Before dedup: {len(df_billboard)} rows")
print(f"After dedup:  {len(df_deduped)} unique songs")

df_deduped.to_csv(DATA_PATH + "billboard_deduped.csv", index=False)

Before dedup: 330087 rows
After dedup:  29681 unique songs


In [10]:
# MusicBrainz

import requests
import time
from tqdm import tqdm

# NOTE: Replacing Spotify with MusicBrainz due to:
# 1. Spotify audio_features endpoint returns HTTP 403 for new apps since late 2024
# 2. Spotify search hit rate limits after ~550 requests
# MusicBrainz is free, no API key required, better coverage for non-Western and older music

HEADERS = {"User-Agent": "MusicAndMemory/1.0 (your@email.com)"}

def search_musicbrainz(title, artist):
    try:
        url = "https://musicbrainz.org/ws/2/recording/"
        params = {
            "query": f'recording:"{title}" AND artist:"{artist}"',
            "fmt": "json",
            "limit": 1
        }
        response = requests.get(url, params=params, headers=HEADERS)
        data = response.json()

        if data.get("recordings"):
            rec = data["recordings"][0]
            releases = rec.get("releases", [])
            country = releases[0].get("country") if releases else None
            date = releases[0].get("date") if releases else None

            return {
                "mb_id": rec.get("id"),
                "mb_title": rec.get("title"),
                "mb_artist": rec["artist-credit"][0]["name"] if rec.get("artist-credit") else None,
                "mb_country": country,
                "mb_date": date,
                "mb_score": rec.get("score"),
            }
    except Exception as e:
        print(f"Error searching for {title} by {artist}: {e}")
    return None

# Test first
test = search_musicbrainz("Hello", "Adele")
print(test)

{'mb_id': 'f9c7447b-6e87-4614-81ec-c52d9d4b1d18', 'mb_title': 'Hello', 'mb_artist': 'Adele', 'mb_country': 'GB', 'mb_date': '2016-06-25', 'mb_score': 100}


In [12]:
# MusicBrainz will take 9 hours to run the full loop, trying to only pull relevant times

print(f"Year range: {df_deduped['year'].min()} to {df_deduped['year'].max()}")
print(df_deduped['year'].value_counts().sort_index())

Year range: 1958 to 2021
year
1958    280
1959    563
1960    586
1961    687
1962    670
       ... 
2017    463
2018    594
2019    509
2020    676
2021    647
Name: count, Length: 64, dtype: int64


In [14]:
# MusicBrainz - filtered approach

# Dementia patients in care today are roughly 70-90 years old
# Born approximately 1935-1955
# Reminiscence bump (ages 15-25) = roughly 1950-1980
# Adding a buffer decade on each side for safety

YEAR_MIN = 1950
YEAR_MAX = 1990

df_filtered = df_deduped[
    (df_deduped['year'] >= YEAR_MIN) &
    (df_deduped['year'] <= YEAR_MAX)
].reset_index(drop=True)

print(f"Full dataset: {len(df_deduped)} songs")
print(f"Filtered to {YEAR_MIN}-{YEAR_MAX}: {len(df_filtered)} songs")
print(f"\nSongs per decade:")
df_filtered['decade'] = (df_filtered['year'] // 10) * 10
print(df_filtered['decade'].value_counts().sort_index())

Full dataset: 29681 songs
Filtered to 1950-1990: 17470 songs

Songs per decade:
decade
1950     843
1960    6852
1970    5279
1980    4115
1990     381
Name: count, dtype: int64


In [ ]:
# running the filtered MusicBrainz loop

# sample top 500 songs per decade for manageable runtime
df_filtered = (
    df_filtered
    .sort_values('peak_rank')
    .groupby('decade')
    .head(500)
    .reset_index(drop=True)
)
print(f"Sampled dataset: {len(df_filtered)} songs")

# Run MusicBrainz on sampled set
mb_data = []
batch_size = 500

for i, row in tqdm(df_filtered.iterrows(), total=len(df_filtered)):
    result = search_musicbrainz(row["song"], row["artist"])
    if result:
        result["billboard_index"] = i
        mb_data.append(result)

    time.sleep(1.1)

    if (i + 1) % batch_size == 0:
        pd.DataFrame(mb_data).to_csv(
            DATA_PATH + f"mb_partial_{i+1}.csv", index=False
        )
        print(f"Saved progress at row {i+1}")

df_mb = pd.DataFrame(mb_data)
print(f"Matched {len(df_mb)} / {len(df_filtered)} songs")
df_mb.to_csv(DATA_PATH + "mb_enriched.csv", index=False)

Sampled dataset: 2381 songs


 21%|██        | 500/2381 [13:53<52:07,  1.66s/it]

Saved progress at row 500


 42%|████▏     | 1000/2381 [27:53<40:03,  1.74s/it]

Saved progress at row 1000


 56%|█████▋    | 1340/2381 [37:24<29:07,  1.68s/it]